# 🔍 Recherche de Meilleurs Hyperparamètres pour DistilBERT

## 🛠️ Configuration et Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score
from transformers import TFDistilBertForSequenceClassification, DistilBertTokenizerFast
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset
import tensorflow as tf

tf.random.set_seed(42)
print("✅ TensorFlow:", tf.__version__)

## 📥 Chargement des Données

In [ ]:
DATA_DIR = "./data"
for split in ["train.csv", "val.csv", "test.csv"]:
    path = os.path.join(DATA_DIR, split)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Le fichier {path} est introuvable.")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
print(f"📊 Données chargées - Train: {len(train_df)} | Val: {len(val_df)}")

## 🧪 Définition des Hyperparamètres à Tester

In [3]:
param_grid = {
    "learning_rates": [2e-5, 3e-5],
    "batch_sizes": [32, 64],
    "epochs": [5, 10]
}

## 🧼 Préparation des Données

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(texts, labels):
    tokens = tokenizer(texts.tolist(), truncation=True, padding=True, return_tensors="tf")
    return Dataset.from_tensor_slices((dict(tokens), labels))

train_dataset = tokenize(train_df["clean_text"], train_df["label"])
val_dataset = tokenize(val_df["clean_text"], val_df["label"])

## 🧠 Fonction d’Entraînement

In [5]:
def train_and_evaluate(lr, batch_size, epochs):
    model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=1)
    optimizer = Adam(learning_rate=lr)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.BinaryCrossentropy(from_logits=True), metrics=["accuracy"])
    history = model.fit(train_dataset.batch(batch_size), epochs=epochs, validation_data=val_dataset.batch(batch_size), verbose=0)

    val_preds = model.predict(val_dataset.batch(batch_size)).logits
    val_preds = tf.sigmoid(val_preds).numpy().flatten()
    val_preds = (val_preds > 0.5).astype(int)

    f1 = f1_score(val_df["label"], val_preds)
    acc = accuracy_score(val_df["label"], val_preds)
    return f1, acc

## 🧪 Lancement du Grid Search

In [ ]:
results = []
for lr in param_grid["learning_rates"]:
    for bs in param_grid["batch_sizes"]:
        for ep in param_grid["epochs"]:
            start = time.time()
            f1, acc = train_and_evaluate(lr, bs, ep)
            duration = time.time() - start
            results.append({"lr": lr, "batch_size": bs, "epochs": ep, "f1": f1, "accuracy": acc, "time": duration})
            print(f"✅ lr={lr}, bs={bs}, ep={ep} -> F1: {f1:.3f}, Acc: {acc:.3f}, Time: {duration:.1f}s")

results_df = pd.DataFrame(results)

## 📊 Visualisation des Résultats

In [ ]:
pivot = results_df.pivot_table(index="batch_size", columns="lr", values="f1", aggfunc="max")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis")
plt.title("F1 Score selon lr et batch_size")
plt.show()

results_df.plot(x="epochs", y=["f1", "accuracy"], kind="bar", figsize=(10, 5))
plt.title("Comparaison des scores F1 / Accuracy")
plt.ylabel("Score")
plt.grid(True)
plt.show()

## 📌 Conclusion

L'exploration de l’espace d’hyperparamètres a permis d’identifier les valeurs suivantes comme les plus efficaces
pour notre tâche de classification de sentiments sur des tweets liés au Bitcoin :

🔁 Epochs : 10
→ Un bon compromis entre convergence et surapprentissage.

📦 Batch size : 32
→ Suffisamment élevé pour un apprentissage stable sans dépasser la mémoire disponible.

📉 Learning rate : 2e-5
→ Donne les meilleures performances de validation, avec une courbe de perte régulière.

🎯 Ces paramètres ont conduit à une F1-score et une accuracy significativement meilleures que les autres combinaisons testées, sans allonger excessivement le temps d’entraînement.
